###  import  the  libraries

In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder   
 

pd.set_option("display.max_columns", None)
print("Libraries loaded ✅")

Libraries loaded ✅


###  load  and re clean  the  data sets    

In [32]:
df = pd.read_csv("../../data/raw/telco_churn.csv")

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"]  = df["TotalCharges"].fillna(df["TotalCharges"].median())

customer_ids = df["customerID"]  # keep for later reference in dashboard
df  =  df.drop(columns=["customerID"])

print(f"Shape: {df.shape}")
df.head()

Shape: (7043, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


###  label encoder  -traget variable    

In [33]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df["Churn"].value_counts()    

Churn
0    5174
1    1869
Name: count, dtype: int64

### Feature engineering: tenure buckets  

In [34]:
def tenure_bucket(t):
    if t <= 12:
        return "0-1yr"
    elif t <= 24:
        return "1-2yr"
    elif t <= 48:
        return "2-4yr"
    else:
        return "4yr+"

df["TenureGroup"] = df["tenure"].apply(tenure_bucket)
df["TenureGroup"].value_counts()

TenureGroup
4yr+     2239
0-1yr    2186
2-4yr    1594
1-2yr    1024
Name: count, dtype: int64

### Feature engineering: charge-based ratios   

In [35]:
# Average charge per month across their whole lifetime vs current monthly charge
df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

# Is customer paying more than their historical average? (recent price hike signal)
df["ChargeIncreaseFlag"] = (df["MonthlyCharges"] > df["AvgMonthlySpend"]).astype(int)

df[["tenure", "MonthlyCharges", "TotalCharges", "AvgMonthlySpend", "ChargeIncreaseFlag"]].head()

,tenure,MonthlyCharges,TotalCharges,AvgMonthlySpend,ChargeIncreaseFlag
0,1,29.85,29.85,29.850000,0
1,34,56.95,1889.50,55.573529,1
2,2,53.85,108.15,54.075000,0
3,45,42.30,1840.75,40.905556,1
4,2,70.70,151.65,75.825000,0


###  Feature engineering: total number of subscribed services     

In [36]:
service_cols = ["PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
                 "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

def count_services(row):
    count = 0
    for col in service_cols:
        if row[col] not in ["No", "No internet service", "No phone service"]:
            count += 1
    return count

df["TotalServices"] = df.apply(count_services, axis=1)
df["TotalServices"].value_counts().sort_index()

TotalServices
1    1264
2     859
3     846
4     965
5     922
6     908
7     676
8     395
9     208
Name: count, dtype: int64

### dentify categorical vs numerical columns   

In [37]:
categorical_cols = df.select_dtypes(include="object").columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols.remove("Churn")  # target, not a feature

print("Categorical:", categorical_cols)
print("\nNumerical:", numerical_cols)   

Categorical: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup']

Numerical: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'TotalServices']


In [38]:
###  get   the  values  count  of  the  object variables   
for cols  in  categorical_cols   :   
    print(df[cols].value_counts())    
    print()  ###  new  line 

gender
Male      3555
Female    3488
Name: count, dtype: int64

Partner
No     3641
Yes    3402
Name: count, dtype: int64

Dependents
No     4933
Yes    2110
Name: count, dtype: int64

PhoneService
Yes    6361
No      682
Name: count, dtype: int64

MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

DeviceProtection
No                     3095
Yes                    2422
No internet service    1526
Name: count, dtype: int64

TechSupport
No                     3473
Yes                    2044
No internet service    1526
Name: count, dtype: int64

StreamingTV
No                 

### Categorize columns by encoding type   

In [39]:
# Binary columns (2 categories, Yes/No or similar) -> map manually
binary_cols = ["gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling"]

# Ordinal columns (natural order exists) -> OrdinalEncoder
ordinal_cols = ["Contract", "TenureGroup"]

# Nominal columns (no order, 3+ categories) -> OneHotEncoder
nominal_cols = ["MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
                "PaymentMethod"]

print("Binary:", binary_cols)
print("Ordinal:", ordinal_cols)
print("Nominal:", nominal_cols)

Binary: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Ordinal: ['Contract', 'TenureGroup']
Nominal: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']


### Binary encoding using map   

In [40]:
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})
df["Partner"] = df["Partner"].map({"Yes": 1, "No": 0})
df["Dependents"] = df["Dependents"].map({"Yes": 1, "No": 0})
df["PhoneService"] = df["PhoneService"].map({"Yes": 1, "No": 0})
df["PaperlessBilling"] = df["PaperlessBilling"].map({"Yes": 1, "No": 0})

df[binary_cols].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,0,1,0,0,1
1,1,0,0,1,0
2,1,0,0,1,1
3,1,0,0,0,0
4,0,0,0,1,1


###  ordinal encoder   

In [41]:
from sklearn.preprocessing import OrdinalEncoder

# Explicit category order matters here — sklearn won't infer "logical" order on its own
contract_order = ["Month-to-month", "One year", "Two year"]
tenure_order = ["0-1yr", "1-2yr", "2-4yr", "4yr+"]

ordinal_encoder = OrdinalEncoder(categories=[contract_order, tenure_order])
df[ordinal_cols] = ordinal_encoder.fit_transform(df[ordinal_cols])

df[ordinal_cols].head()

,Contract,TenureGroup
0,0.0,0.0
1,1.0,2.0
2,0.0,0.0
3,1.0,2.0
4,0.0,0.0


###  one  hot encoding   

In [42]:
###  one  hot  encoder    

from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore")
onehot_encoded = onehot_encoder.fit_transform(df[nominal_cols])

onehot_feature_names = onehot_encoder.get_feature_names_out(nominal_cols)
onehot_df = pd.DataFrame(onehot_encoded, columns=onehot_feature_names, index=df.index)

onehot_df.head()

,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [43]:
### combine  everythings  

# Drop original nominal columns, merge one-hot encoded columns back in
df_encoded = df.drop(columns=nominal_cols)
df_encoded = pd.concat([df_encoded, onehot_df], axis=1)

print(f"Final shape: {df_encoded.shape}")
df_encoded.head()

Final shape: (7043, 34)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,Contract,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,TenureGroup,AvgMonthlySpend,ChargeIncreaseFlag,TotalServices,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,OnlineBackup_No internet service,OnlineBackup_Yes,DeviceProtection_No internet service,DeviceProtection_Yes,TechSupport_No internet service,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0.0,1,29.85,29.85,0,0.0,29.850000,0,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,34,1,1.0,0,56.95,1889.50,0,2.0,55.573529,1,4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1,0,0,0,2,1,0.0,1,53.85,108.15,1,0.0,54.075000,0,4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,45,0,1.0,0,42.30,1840.75,0,2.0,40.905556,1,4,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,0,0,0,2,1,0.0,1,70.70,151.65,1,0.0,75.825000,0,2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


###    detect  the  null   values   

In [44]:
print("Nulls remaining:", df_encoded.isnull().sum().sum())
print("Data types:\n", df_encoded.dtypes.value_counts())

Nulls remaining: 0
Data types:
 float64    24
int64       9
int32       1
Name: count, dtype: int64


In [45]:
###  no  any    missing  values   

### save  the  processed  data     

In [46]:
import os
os.makedirs("../../data/processed", exist_ok=True)

df_encoded.to_csv("../../data/processed/churn_cleaned.csv", index=False)
print("Saved to data/processed/churn_cleaned.csv ✅")   

Saved to data/processed/churn_cleaned.csv ✅


### Save feature column order + encoders  

In [47]:
### for  the api   
import json
import joblib

feature_columns = [col for col in df_encoded.columns if col != "Churn"]
with open("../models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

# Save the fitted encoders themselves so the FastAPI backend can transform new customer input consistently
joblib.dump(ordinal_encoder, "../models/ordinal_encoder.pkl")
joblib.dump(onehot_encoder, "../models/onehot_encoder.pkl")

print(f"Saved {len(feature_columns)} feature names and both encoders ✅")  

Saved 33 feature names and both encoders ✅


### summary   

## Feature Engineering Summary
- **Binary** (gender, Partner, Dependents, PhoneService, PaperlessBilling) → manually mapped to 0/1
- **Ordinal** (Contract, TenureGroup) → `OrdinalEncoder` with explicit logical order
- **Nominal** (MultipleLines, InternetService, OnlineSecurity, etc., PaymentMethod) → `OneHotEncoder`
- Saved fitted encoders (`ordinal_encoder.pkl`, `onehot_encoder.pkl`) — required so the FastAPI backend encodes new customer input **the same way** as training data
- Final dataset saved to `data/processed/churn_cleaned.csv`  